In [0]:
SELECT *
FROM read_files('/Volumes/idp/default/final_project')

In [0]:
CREATE OR REPLACE TABLE parsed_data AS 
  SELECT 
    path,
    ai_parse_document(content) AS parsed_content
  FROM read_files('/Volumes/idp/default/final_project')

In [0]:
CREATE OR REPLACE TABLE pretty_data AS
  SELECT
    path,
    concat_ws('\n',
      transform(cast(parsed_content:document:elements AS ARRAY<VARIANT>), e -> coalesce(try_cast(e:content AS STRING), ''))
    ) AS doc_text
  FROM parsed_data

In [0]:
CREATE OR REPLACE TABLE classified_data AS  
  SELECT 
    *,
    ai_classify(doc_text, ARRAY('Invoice', 'Purchase Order', 'Receipt', 'Other')) AS document_type
  FROM pretty_data

In [0]:
CREATE OR REPLACE TABLE invoice_data AS  
  SELECT 
    *,
    ai_extract(doc_text,
      ARRAY(
        'Vendor_Name',
        'Invoice_Number',
        'Invoice_Date',
        'Due_Date',
        'Payment_Method',
        'Total')) AS extracted
  FROM classified_data
  WHERE document_type = 'Invoice'

In [0]:
CREATE SCHEMA IF NOT EXISTS idp.finance

In [0]:
CREATE OR REPLACE TABLE idp.finance.invoices AS
  SELECT
    path,
    extracted.Vendor_Name AS vendor,
    extracted.Invoice_Number AS invoice_number,
    extracted.Invoice_Date AS invoice_date,
    extracted.Due_Date AS due_date,
    extracted.Payment_Method AS payment_method,
    extracted.Total AS total
  FROM invoice_data

In [0]:
SELECT *
FROM idp.finance.invoices

In [0]:
CREATE OR REPLACE TABLE purchase_order_data AS  
  SELECT 
    *,
    ai_extract(doc_text,
      ARRAY(
        'Merchant_Name',
        'PO_Number',
        'Purchase_Order_Date',
        'Total')) AS extracted
  FROM classified_data
  WHERE document_type = 'Purchase Order'

In [0]:
CREATE OR REPLACE TABLE idp.finance.purchase_order AS
  SELECT
    path,
    extracted.Merchant_Name AS merchant,
    extracted.PO_Number AS po_number,
    extracted.Purchase_Order_Date AS po_date,
    extracted.Total AS total
  FROM purchase_order_data

In [0]:
CREATE OR REPLACE TABLE receipt_data AS  
  SELECT 
    *,
    ai_extract(doc_text,
      ARRAY(
        'Merchant_Name',
        'Receipt_Number',
        'Receipt_Date',
        'Total')) AS extracted
  FROM classified_data
  WHERE document_type = 'Receipt'

In [0]:
CREATE OR REPLACE TABLE idp.finance.receipt AS
  SELECT
    path,
    extracted.Merchant_Name AS merchant,
    extracted.Receipt_Number AS receipt_number,
    extracted.Receipt_Date AS receipt_date,
    extracted.Total AS total
  FROM receipt_data